# AML 01 — Data Preparation and Feature Engineering

## Dual-Task Causal NLP Pipeline Overview

In alignment with `DL_AML_Project_Idea.pdf` (Project 1: Causal NLP Reasoning — *Neuro-Symbolic Causal Inference via Language Model Knowledge Extraction and Bayesian Networks*), this notebook implements the foundational data preparation and feature engineering pipeline across two strictly separated tasks:

1. **Primary Relation Extraction Dataset: BioCause**
   - **Corpus Source:** National Centre for Text Mining (NaCTeM) / BioNLP 2011 Shared Task Infectious Diseases (ID) enriched with causal annotations ([Mihăilă et al., 2013](https://doi.org/10.1186/1471-2105-14-2)).
   - **Licensing & Attribution:** Full texts originate from PubMed Central Open Access subset. Annotations are licensed by NaCTeM under Creative Commons Attribution-NonCommercial-ShareAlike 3.0 Unported (CC BY-NC-SA 3.0). The corpus and derived raw texts are strictly kept in git-ignored local directories (`data/raw/` and `data/processed/`) and never redistributed.
   - **Structure & Scope:** The official distributed archive contains 20 PubMed Central full-text articles across 198 section files (`.txt` and stand-off `.ann`), containing 851 gold causal relation annotations, 1,526 causal argument spans, and 381 distinct causal trigger expressions.
   - **Article-Level Group Partitioning:** Splitting is performed strictly by source `article_id` (never by section or sentence) to guarantee complete document-level isolation: 12 train articles, 4 validation articles, 4 test articles. Because the corpus contains only 20 source articles, test metrics are sensitive to document-level variance, a documented limitation addressed across the pipeline.
   - **Standardized Relation Schema:** Exports exact source character offsets, document/section IDs, trigger metadata, cause/effect spans, and candidate argument pairs.

2. **Supplementary Causal Entailment Benchmark: Corr2Cause**
   - Preserves the existing benchmark as a supplementary causal reasoning experiment ([Jin et al., 2024](https://arxiv.org/abs/2306.05836)).
   - Distinct from relation extraction: Corr2Cause tests whether a formal causal premise entails an abstract hypothesis (binary classification). It does **not** contain gold relation extraction triples or character-level span annotations.
   - Computes 19 engineered features over premise and hypothesis text for classical baseline comparison in AML_04.

---


## 1. Configuration, Imports, and Path Resolution

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import glob
from pathlib import Path
from typing import Dict, List, Tuple, Set, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def find_repo_root() -> Path:
    """Locate the repository root using stable anchor files (README.md and 00_setup.ipynb).
    Searches upward from the current working directory and current file location.
    Fails with an actionable RuntimeError if the repository root cannot be identified.
    """
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    if "__file__" in globals():
        fpath = Path(__file__).resolve().parent
        candidates.extend([fpath, *fpath.parents])
    for candidate in candidates:
        if (candidate / "00_setup.ipynb").is_file() and (candidate / "README.md").is_file():
            return candidate.resolve()
    raise RuntimeError(
        f"Repository root containing 00_setup.ipynb and README.md could not be identified "
        f"from current working directory: {cwd}. Please ensure execution starts within the repository."
    )

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
BIOCAUSE_RAW_DIR = RAW_DIR / "biocause" / "BioCause_corpus"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Repository Root: {REPO_ROOT.name}")
print(f"✓ Raw Data Dir   : {RAW_DIR.relative_to(REPO_ROOT)}")
print(f"✓ Processed Dir  : {PROCESSED_DIR.relative_to(REPO_ROOT)}")

✓ Repository Root: causal-nlp-reasoning
✓ Raw Data Dir   : data/raw
✓ Processed Dir  : data/processed


## 2. Primary Task: BioCause Corpus Consumption and Validation

We verify the local availability of the official BioCause corpus prepared by `00_setup.ipynb`.
If not already present, we instruct the user to run `00_setup.ipynb` first.

We inspect the stand-off annotation structure (`.ann` and `.txt`), validate text span offsets against actual file contents, and extract causal events, triggers, and arguments.


In [2]:
def verify_biocause_available() -> Path:
    """Verify that BioCause corpus has been acquired and validated by 00_setup.ipynb."""
    target_dir = BIOCAUSE_RAW_DIR
    if not target_dir.exists() or len(list(target_dir.glob("*.ann"))) != 198:
        # Fallback check directly in raw/biocause
        if (RAW_DIR / "biocause").exists() and len(list((RAW_DIR / "biocause").glob("*.ann"))) == 198:
            target_dir = RAW_DIR / "biocause"
        else:
            raise FileNotFoundError(
                f"BioCause corpus not found or incomplete at {BIOCAUSE_RAW_DIR.relative_to(REPO_ROOT)}. "
                "Please run '00_setup.ipynb' first to acquire and validate the BioCause dataset."
            )
    ann_files = sorted(list(target_dir.glob("*.ann")))
    txt_files = sorted(list(target_dir.glob("*.txt")))
    print(f"✓ BioCause corpus located at {target_dir.relative_to(REPO_ROOT)} ({len(ann_files)} .ann, {len(txt_files)} .txt files).")
    return target_dir

BIOCAUSE_RAW_DIR = verify_biocause_available()


✓ BioCause corpus located at data/raw/biocause/BioCause_corpus (198 .ann, 198 .txt files).


### 2.1 BioCause Stand-Off Parser & Character Offset Verification

Stand-off annotation files (`.ann`) specify character spans $[\text{start}, \text{end})$ corresponding to document texts (`.txt`).
We parse:
- **T lines (Terms / Spans / Triggers):** ID, type, character start/end, and text string.
- **E lines (Events / Relations):** Event ID, event trigger ID, and typed arguments (`Evidence`/`Cause` $\to$ `Effect`).

We programmatically verify that every annotation character offset resolves to the exact expected text in the `.txt` file, accounting for standard newline representation normalization.


In [3]:
def parse_biocause_standoff(corpus_path: Path):
    """Parse all .txt and .ann files from BioCause and validate span character offsets.
    Verifies that all spans resolve cleanly against underlying text and explicitly rejects
    discontinuous spans if encountered.
    """
    ann_paths = sorted(corpus_path.glob("*.ann"))
    assert len(ann_paths) == 198, f"Expected 198 .ann files, found {len(ann_paths)}"
    
    parsed_sections = {}
    total_spans_checked = 0
    mismatches = 0
    discontinuous_count = 0
    
    for af in ann_paths:
        sec_id = af.stem
        art_id = sec_id.split("-")[0]
        tf = af.with_suffix(".txt")
        assert tf.exists(), f"Missing text file for {af}"
        
        with open(tf, "r", encoding="utf-8") as f:
            doc_text = f.read()
            
        t_annotations = {}
        e_annotations = []
        
        with open(af, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                if line.startswith("T"):
                    parts = line.split("\t")
                    tid = parts[0]
                    offset_info = parts[1]
                    
                    # Discontinuous span check: BRAT uses semicolon to separate discontinuous fragments
                    if ";" in offset_info:
                        discontinuous_count += 1
                        raise NotImplementedError(
                            f"Discontinuous span detected in {sec_id}: {line}. "
                            "Contiguous span pipeline does not support discontinuous spans."
                        )
                    
                    meta = offset_info.split()
                    ttype = meta[0]
                    s, e = int(meta[1]), int(meta[2])
                    raw_text = parts[2] if len(parts) > 2 else doc_text[s:e]
                    
                    # Verify offset against doc_text
                    actual = doc_text[s:e]
                    total_spans_checked += 1
                    if actual != raw_text and actual.replace("\n", " ") != raw_text:
                        mismatches += 1
                    
                    t_annotations[tid] = {
                        "id": tid, "type": ttype, "start": s, "end": e,
                        "text": raw_text, "actual_text": actual
                    }
                elif line.startswith("E"):
                    parts = line.split("\t")
                    eid = parts[0]
                    tokens = parts[1].split()
                    etype, trig_id = tokens[0].split(":")
                    args = {}
                    for tok in tokens[1:]:
                        r, val = tok.split(":")
                        args[r] = val
                    e_annotations.append({
                        "event_id": eid, "type": etype, "trigger_id": trig_id,
                        "args": args
                    })
                    
        parsed_sections[sec_id] = {
            "article_id": art_id,
            "section_id": sec_id,
            "doc_text": doc_text,
            "terms": t_annotations,
            "events": e_annotations
        }
        
    print(f"Parsed {len(parsed_sections)} section files across 20 PMC articles.")
    print(f"Total spans validated: {total_spans_checked:,} | Offset mismatches: {mismatches}")
    print(f"Discontinuous spans found in BioCause: {discontinuous_count} (Corpus invariant verified: 0 discontinuous spans)")
    assert mismatches == 0, f"Detected {mismatches} unresolved span offset mismatches!"
    assert discontinuous_count == 0, f"Detected unexpected discontinuous spans: {discontinuous_count}"
    print("✓ 100% character offset resolution verified across all document sections.")
    return parsed_sections

biocause_data = parse_biocause_standoff(BIOCAUSE_RAW_DIR)

Parsed 198 section files across 20 PMC articles.
Total spans validated: 12,939 | Offset mismatches: 0
Discontinuous spans found in BioCause: 0 (Corpus invariant verified: 0 discontinuous spans)
✓ 100% character offset resolution verified across all document sections.


## 3. Article-Level Group Splitting (Deterministic & Isolated)

To prevent data leakage, we split exclusively at the **source article level** (`article_id`), never randomly by sentence or section. All sections from a given article belong to the same split.

- **Total Articles in Corpus:** 20 PMC article IDs
- **Train Split (60%):** 12 articles
- **Validation Split (20%):** 4 articles
- **Test Split (20%):** 4 articles

We fix random seed 42 and assert that the intersection between splits is strictly empty.


In [4]:
unique_articles = sorted(list(set(sec["article_id"] for sec in biocause_data.values())))
print(f"Unique source articles in BioCause: {len(unique_articles)}")
print(f"Article IDs: {unique_articles}")

# Deterministic shuffle with seed 42
rng = np.random.RandomState(42)
shuffled_articles = list(unique_articles)
rng.shuffle(shuffled_articles)

train_articles = set(shuffled_articles[:12])
val_articles = set(shuffled_articles[12:16])
test_articles = set(shuffled_articles[16:])

# Strict isolation assertions
assert train_articles.isdisjoint(val_articles), "Train and Validation articles overlap!"
assert train_articles.isdisjoint(test_articles), "Train and Test articles overlap!"
assert val_articles.isdisjoint(test_articles), "Validation and Test articles overlap!"
assert len(train_articles | val_articles | test_articles) == 20, "Missing articles in split assignment!"

print(f"✓ Train articles ({len(train_articles)}): {sorted(train_articles)}")
print(f"✓ Validation articles ({len(val_articles)}): {sorted(val_articles)}")
print(f"✓ Test articles ({len(test_articles)}): {sorted(test_articles)}")
print("✓ Strict group-level article isolation asserted.")


Unique source articles in BioCause: 20
Article IDs: ['PMC1804205', 'PMC1874608', 'PMC1913099', 'PMC1974823', 'PMC2242835', 'PMC2266911', 'PMC2358977', 'PMC2430206', 'PMC2565068', 'PMC2581603', 'PMC2593050', 'PMC2639726', 'PMC2651894', 'PMC2682197', 'PMC2714965', 'PMC2774163', 'PMC2816692', 'PMC2829055', 'PMC2858072', 'PMC2885601']
✓ Train articles (12): ['PMC1804205', 'PMC1874608', 'PMC1913099', 'PMC1974823', 'PMC2266911', 'PMC2565068', 'PMC2639726', 'PMC2682197', 'PMC2774163', 'PMC2816692', 'PMC2829055', 'PMC2858072']
✓ Validation articles (4): ['PMC2242835', 'PMC2581603', 'PMC2651894', 'PMC2885601']
✓ Test articles (4): ['PMC2358977', 'PMC2430206', 'PMC2593050', 'PMC2714965']
✓ Strict group-level article isolation asserted.


## 4. Shared Relation Schema and Table Construction

We construct three standardized tables adhering to the project schema contracts:

1. **`biocause_relations.csv`**:
   The gold standard table of 852 causal relations (851 causal events + 1 branching event).
   Schema: `relation_id`, `dataset`, `article_id`, `section_id`, `trigger_id`, `trigger_text`, `trigger_start`, `trigger_end`, `cause_id`, `cause_text`, `cause_start`, `cause_end`, `effect_id`, `effect_text`, `effect_start`, `effect_end`, `direction`, `is_gold`, `split`.

2. **`biocause_entities.csv`**:
   All 12,939 gold annotated entity and event mentions (Proteins, Organisms, Chemicals, Two-component systems, Argument Spans).
   Used as reference annotations for NER evaluation in `AML_02`.

3. **`biocause_candidates.csv`**:
   The candidate argument pairs for relation extraction and classification in `AML_03` and `AML_04`.
   Generated without using held-out gold labels: pairs of argument spans $(Arg_1, Arg_2)$ occurring within 2,000 characters in the same document section, labeled with binary causality (1 = gold causal relation, 0 = non-causal).


In [5]:
def get_article_split(art_id: str) -> str:
    if art_id in train_articles:
        return "train"
    elif art_id in val_articles:
        return "validation"
    else:
        return "test"

relations_records = []
entities_records = []
candidates_records = []

for sec_id, sec_data in biocause_data.items():
    art_id = sec_data["article_id"]
    split = get_article_split(art_id)
    doc_text = sec_data["doc_text"]
    terms = sec_data["terms"]
    events = sec_data["events"]
    
    # 1. Entities
    for tid, tinfo in terms.items():
        entities_records.append({
            "mention_id": f"{sec_id}_{tid}",
            "dataset": "BioCause",
            "article_id": art_id,
            "section_id": sec_id,
            "entity_type": tinfo["type"],
            "start_char": tinfo["start"],
            "end_char": tinfo["end"],
            "text": tinfo["text"],
            "split": split,
            "is_gold": 1
        })
        
    # 2. Gold causal relations
    gold_causal_pairs = set()
    for ev in events:
        if ev["type"] == "Causality":
            eid = ev["event_id"]
            trig_id = ev["trigger_id"]
            args = ev["args"]
            
            trig_info = terms.get(trig_id, {})
            cid = args.get("Evidence") or args.get("Cause")
            eid_arg = args.get("Effect")
            eid2_arg = args.get("Effect2")
            
            if cid and eid_arg and cid in terms and eid_arg in terms:
                gold_causal_pairs.add((cid, eid_arg))
                relations_records.append({
                    "relation_id": f"{sec_id}_{eid}",
                    "dataset": "BioCause",
                    "article_id": art_id,
                    "section_id": sec_id,
                    "trigger_id": trig_id,
                    "trigger_text": trig_info.get("text", ""),
                    "trigger_start": trig_info.get("start", -1),
                    "trigger_end": trig_info.get("end", -1),
                    "cause_id": cid,
                    "cause_text": terms[cid]["text"],
                    "cause_start": terms[cid]["start"],
                    "cause_end": terms[cid]["end"],
                    "effect_id": eid_arg,
                    "effect_text": terms[eid_arg]["text"],
                    "effect_start": terms[eid_arg]["start"],
                    "effect_end": terms[eid_arg]["end"],
                    "direction": "cause->effect",
                    "is_gold": 1,
                    "split": split
                })
            if cid and eid2_arg and cid in terms and eid2_arg in terms:
                gold_causal_pairs.add((cid, eid2_arg))
                relations_records.append({
                    "relation_id": f"{sec_id}_{eid}_branch2",
                    "dataset": "BioCause",
                    "article_id": art_id,
                    "section_id": sec_id,
                    "trigger_id": trig_id,
                    "trigger_text": trig_info.get("text", ""),
                    "trigger_start": trig_info.get("start", -1),
                    "trigger_end": trig_info.get("end", -1),
                    "cause_id": cid,
                    "cause_text": terms[cid]["text"],
                    "cause_start": terms[cid]["start"],
                    "cause_end": terms[cid]["end"],
                    "effect_id": eid2_arg,
                    "effect_text": terms[eid2_arg]["text"],
                    "effect_start": terms[eid2_arg]["start"],
                    "effect_end": terms[eid2_arg]["end"],
                    "direction": "cause->effect",
                    "is_gold": 1,
                    "split": split
                })
                
    # 3. Candidate pairs (all ordered pairs of argument spans within 2,000 characters)
    arg_tids = [t for t, info in terms.items() if info["type"] in ("Span", "Fact", "Argument")]
    for i in range(len(arg_tids)):
        for j in range(len(arg_tids)):
            if i != j:
                a1 = terms[arg_tids[i]]
                a2 = terms[arg_tids[j]]
                dist = abs((a1["start"] + a1["end"]) / 2 - (a2["start"] + a2["end"]) / 2)
                if dist <= 2000:
                    is_causal = 1 if (arg_tids[i], arg_tids[j]) in gold_causal_pairs else 0
                    if a1["end"] <= a2["start"]:
                        between_text = doc_text[a1["end"]:a2["start"]]
                        a1_before_a2 = 1
                    elif a2["end"] <= a1["start"]:
                        between_text = doc_text[a2["end"]:a1["start"]]
                        a1_before_a2 = 0
                    else:
                        between_text = ""
                        a1_before_a2 = 1 if a1["start"] <= a2["start"] else 0
                        
                    candidates_records.append({
                        "candidate_id": f"{sec_id}_{arg_tids[i]}_{arg_tids[j]}",
                        "dataset": "BioCause",
                        "article_id": art_id,
                        "section_id": sec_id,
                        "arg1_id": arg_tids[i],
                        "arg1_text": a1["text"],
                        "arg1_start": a1["start"],
                        "arg1_end": a1["end"],
                        "arg2_id": arg_tids[j],
                        "arg2_text": a2["text"],
                        "arg2_start": a2["start"],
                        "arg2_end": a2["end"],
                        "char_distance": round(dist, 1),
                        "arg1_before_arg2": a1_before_a2,
                        "between_text": between_text,
                        "label": is_causal,
                        "split": split
                    })

df_biocause_relations = pd.DataFrame(relations_records)
df_biocause_entities = pd.DataFrame(entities_records)
df_biocause_candidates = pd.DataFrame(candidates_records)

# Assert candidate generation integrity invariants
assert len(df_biocause_candidates) == df_biocause_candidates["candidate_id"].nunique(), "Duplicate candidate_ids detected!"
assert (df_biocause_candidates["arg1_id"] == df_biocause_candidates["arg2_id"]).sum() == 0, "Self-loop candidate pairs detected!"

cand_set = set(zip(df_biocause_candidates["section_id"], df_biocause_candidates["arg1_id"], df_biocause_candidates["arg2_id"]))
gold_set = set(zip(df_biocause_relations["section_id"], df_biocause_relations["cause_id"], df_biocause_relations["effect_id"]))
missed_gold = gold_set - cand_set
assert len(missed_gold) == 0, f"Candidate generation missed {len(missed_gold)} gold relations: {missed_gold}"

print(f"✓ BioCause Relations : {len(df_biocause_relations):,} rows across {len(df_biocause_relations.columns)} columns")
print(f"✓ BioCause Entities  : {len(df_biocause_entities):,} rows across {len(df_biocause_entities.columns)} columns")
print(f"✓ BioCause Candidates: {len(df_biocause_candidates):,} rows across {len(df_biocause_candidates.columns)} columns")
print(f"✓ Candidate Coverage : {len(gold_set)}/{len(gold_set)} (100.00%) gold relations captured in candidate pairs (0 missed).")

✓ BioCause Relations : 852 rows across 19 columns
✓ BioCause Entities  : 12,939 rows across 10 columns
✓ BioCause Candidates: 13,408 rows across 17 columns
✓ Candidate Coverage : 852/852 (100.00%) gold relations captured in candidate pairs (0 missed).


In [6]:
print("=" * 70)
print("BIOCAUSE SPLIT SUMMARY")
print("=" * 70)
summary_df = pd.DataFrame({
    "Split": ["train", "validation", "test", "TOTAL"],
    "Articles": [len(train_articles), len(val_articles), len(test_articles), 20],
    "Sections": [
        df_biocause_relations[df_biocause_relations["split"] == "train"]["section_id"].nunique(),
        df_biocause_relations[df_biocause_relations["split"] == "validation"]["section_id"].nunique(),
        df_biocause_relations[df_biocause_relations["split"] == "test"]["section_id"].nunique(),
        df_biocause_relations["section_id"].nunique()
    ],
    "Relations": [
        (df_biocause_relations["split"] == "train").sum(),
        (df_biocause_relations["split"] == "validation").sum(),
        (df_biocause_relations["split"] == "test").sum(),
        len(df_biocause_relations)
    ],
    "Candidates": [
        (df_biocause_candidates["split"] == "train").sum(),
        (df_biocause_candidates["split"] == "validation").sum(),
        (df_biocause_candidates["split"] == "test").sum(),
        len(df_biocause_candidates)
    ],
    "Causal Rate (%)": [
        round((df_biocause_candidates[df_biocause_candidates["split"] == "train"]["label"].mean()) * 100, 2),
        round((df_biocause_candidates[df_biocause_candidates["split"] == "validation"]["label"].mean()) * 100, 2),
        round((df_biocause_candidates[df_biocause_candidates["split"] == "test"]["label"].mean()) * 100, 2),
        round((df_biocause_candidates["label"].mean()) * 100, 2)
    ]
})
display(summary_df)

# Export processed BioCause tables to data/processed/
rel_out = PROCESSED_DIR / "biocause_relations.csv"
ent_out = PROCESSED_DIR / "biocause_entities.csv"
cand_out = PROCESSED_DIR / "biocause_candidates.csv"

df_biocause_relations.to_csv(rel_out, index=False)
df_biocause_entities.to_csv(ent_out, index=False)
df_biocause_candidates.to_csv(cand_out, index=False)

print(f"✓ Exported: {rel_out.relative_to(REPO_ROOT)} ({rel_out.stat().st_size / 1024:.1f} KB)")
print(f"✓ Exported: {ent_out.relative_to(REPO_ROOT)} ({ent_out.stat().st_size / 1024:.1f} KB)")
print(f"✓ Exported: {cand_out.relative_to(REPO_ROOT)} ({cand_out.stat().st_size / (1024*1024):.2f} MB)")

BIOCAUSE SPLIT SUMMARY


,Split,Articles,Sections,Relations,Candidates,Causal Rate (%)
0,train,12,121,621,10086,6.16
1,validation,4,33,131,1914,6.84
2,test,4,31,100,1408,7.10
3,TOTAL,20,185,852,13408,6.35


✓ Exported: data/processed/biocause_relations.csv (334.9 KB)
✓ Exported: data/processed/biocause_entities.csv (1644.6 KB)
✓ Exported: data/processed/biocause_candidates.csv (12.90 MB)


## 5. Supplementary Benchmark: Corr2Cause Ingestion

Corr2Cause serves as a supplementary benchmark testing pure formal causal entailment. We load `train_val.csv` (206,810 rows) and `test.csv` (1,162 rows), reconstruct the original training/validation partitions, and compute the 19 lexical and structural features.


In [7]:
EXPECTED_COLUMNS = ["input", "label", "template", "num_variables"]
ORIGINAL_TRAIN_SIZE = 205734
ORIGINAL_VAL_SIZE = 1076
EXPECTED_TRAIN_VAL_SIZE = ORIGINAL_TRAIN_SIZE + ORIGINAL_VAL_SIZE
EXPECTED_TEST_SIZE = 1162

def load_corr2cause(name: str, expected_size: int):
    path = PROCESSED_DIR / f"{name}.csv"
    if not path.exists():
        path = RAW_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}.csv in processed and raw dirs.")
    df = pd.read_csv(path)
    print(f"Loaded {name}.csv: {len(df):,} rows × {len(df.columns)} columns")
    for col in EXPECTED_COLUMNS:
        assert col in df.columns, f"Missing expected column {col}"
    assert len(df) == expected_size, f"Size mismatch: expected {expected_size}, got {len(df)}"
    return df

df_tv_c2c = load_corr2cause("train_val", EXPECTED_TRAIN_VAL_SIZE)
df_test_c2c = load_corr2cause("test", EXPECTED_TEST_SIZE)

# Reconstruct original train / validation splits
split_labels = ["train"] * ORIGINAL_TRAIN_SIZE + ["validation"] * ORIGINAL_VAL_SIZE
df_tv_c2c["original_split"] = split_labels
print(f"✓ Split reconstruction: {df_tv_c2c['original_split'].value_counts().to_dict()}")


Loaded train_val.csv: 206,810 rows × 4 columns
Loaded test.csv: 1,162 rows × 4 columns
✓ Split reconstruction: {'train': 205734, 'validation': 1076}


## 6. Corr2Cause Text Preprocessing and Feature Engineering

We extract premise and hypothesis components from Corr2Cause inputs and compute 19 defensible lexical, syntactic, and structural features.


In [8]:
PREMISE_HYPOTHESIS_PATTERN = re.compile(
    r"^Premise:\s*(.+?)\s*\nHypothesis:\s*(.+)$",
    re.DOTALL
)

def extract_premise_hypothesis(text: str) -> Tuple[str, str]:
    match = PREMISE_HYPOTHESIS_PATTERN.match(text)
    if not match:
        raise ValueError(f"Failed to match premise/hypothesis in text: {text[:80]}...")
    return match.group(1).strip(), match.group(2).strip()

_SINGLE_VAR_RE = re.compile(r"\b([A-Z])\b")
_CAUSAL_KEYWORDS_RE = re.compile(
    r"\b(?:causes|caused|causing|cause|is a cause of|produces|produced|produces|leads to|led to)\b",
    re.IGNORECASE
)

def count_sentences(text: str) -> int:
    return max(1, len([s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]))

def extract_variable_names(text: str) -> List[str]:
    # Match single uppercase letters that represent causal graph nodes
    raw_matches = _SINGLE_VAR_RE.findall(text)
    return sorted(list(set(raw_matches)))

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    pairs = [extract_premise_hypothesis(t) for t in df["input"]]
    df["premise"] = [p[0] for p in pairs]
    df["hypothesis"] = [p[1] for p in pairs]
    
    # 1. Text length features
    df["input_char_count"] = df["input"].str.len()
    df["input_word_count"] = df["input"].str.split().str.len()
    df["premise_char_count"] = df["premise"].str.len()
    df["premise_word_count"] = df["premise"].str.split().str.len()
    df["premise_sentence_count"] = df["premise"].apply(count_sentences)
    df["hypothesis_char_count"] = df["hypothesis"].str.len()
    df["hypothesis_word_count"] = df["hypothesis"].str.split().str.len()
    
    # 2. Premise relational cues
    df["n_correlations"] = df["premise"].str.count(r"(?i:correlated with)")
    df["n_independences"] = df["premise"].str.count(r"(?i:independent of)")
    df["n_given_clauses"] = df["premise"].str.count(r"(?i:given that)")
    df["has_however"] = df["premise"].str.contains(r"(?i:\bhowever\b)").astype(int)
    
    # 3. Ratio features
    df["premise_hyp_word_ratio"] = df["premise_word_count"] / df["hypothesis_word_count"].clip(lower=1)
    df["indep_corr_ratio"] = df["n_independences"] / (df["n_correlations"] + 1)
    df["hyp_causal_keyword_count"] = df["hypothesis"].str.count(_CAUSAL_KEYWORDS_RE)
    
    # 4. Symbolic variable features
    p_vars = [extract_variable_names(p) for p in df["premise"]]
    h_vars = [extract_variable_names(h) for h in df["hypothesis"]]
    df["n_premise_variables"] = [len(v) for v in p_vars]
    df["n_hypothesis_variables"] = [len(v) for v in h_vars]
    df["variable_overlap"] = [len(set(pv) & set(hv)) for pv, hv in zip(p_vars, h_vars)]
    df["all_hyp_vars_in_premise"] = [int(set(hv).issubset(set(pv))) for pv, hv in zip(p_vars, h_vars)]
    df["n_total_relations"] = df["n_correlations"] + df["n_independences"]
    
    return df

print("Engineering features for Corr2Cause train_val...")
df_tv_c2c = engineer_features(df_tv_c2c)
print("Engineering features for Corr2Cause test...")
df_test_c2c = engineer_features(df_test_c2c)
print(f"✓ Total columns after feature engineering: {len(df_tv_c2c.columns)}")


Engineering features for Corr2Cause train_val...


Engineering features for Corr2Cause test...
✓ Total columns after feature engineering: 26


In [9]:
FEATURE_COLS = [
    "input_char_count", "input_word_count",
    "premise_char_count", "premise_word_count", "premise_sentence_count",
    "hypothesis_char_count", "hypothesis_word_count",
    "n_correlations", "n_independences", "n_given_clauses",
    "has_however", "premise_hyp_word_ratio", "indep_corr_ratio",
    "hyp_causal_keyword_count",
    "n_premise_variables", "n_hypothesis_variables",
    "variable_overlap", "all_hyp_vars_in_premise",
    "n_total_relations",
]

assert len(FEATURE_COLS) == 19
assert "label" not in FEATURE_COLS
assert "original_split" not in FEATURE_COLS

# Export feature-enriched Corr2Cause datasets
tv_feat_out = PROCESSED_DIR / "train_val_features.csv"
test_feat_out = PROCESSED_DIR / "test_features.csv"

df_tv_c2c.to_csv(tv_feat_out, index=False)
df_test_c2c.to_csv(test_feat_out, index=False)

print(f"✓ Exported: {tv_feat_out.relative_to(REPO_ROOT)} ({tv_feat_out.stat().st_size / (1024*1024):.1f} MB)")
print(f"✓ Exported: {test_feat_out.relative_to(REPO_ROOT)} ({test_feat_out.stat().st_size / 1024:.1f} KB)")


✓ Exported: data/processed/train_val_features.csv (740.4 MB)
✓ Exported: data/processed/test_features.csv (2333.1 KB)


## 7. Pipeline Verification and Integrity Audits

We reload the exported processed datasets and execute assertions validating:
1. Article-level split isolation (zero article overlap).
2. BioCause span character resolution (100% valid text spans).
3. Non-null schema fields and correct row counts.
4. Absence of target leakage.


In [10]:
# 1. Reload BioCause tables
df_rel_re = pd.read_csv(PROCESSED_DIR / "biocause_relations.csv", keep_default_na=False)
df_ent_re = pd.read_csv(PROCESSED_DIR / "biocause_entities.csv", keep_default_na=False)
df_cand_re = pd.read_csv(PROCESSED_DIR / "biocause_candidates.csv", keep_default_na=False)

# 2. Assert split isolation across reloaded BioCause tables
for table_name, df_check in [("relations", df_rel_re), ("entities", df_ent_re), ("candidates", df_cand_re)]:
    train_arts_re = set(df_check[df_check["split"] == "train"]["article_id"].unique())
    val_arts_re = set(df_check[df_check["split"] == "validation"]["article_id"].unique())
    test_arts_re = set(df_check[df_check["split"] == "test"]["article_id"].unique())
    assert train_arts_re.isdisjoint(val_arts_re), f"Leakage between train and val in {table_name}"
    assert train_arts_re.isdisjoint(test_arts_re), f"Leakage between train and test in {table_name}"
    assert val_arts_re.isdisjoint(test_arts_re), f"Leakage between val and test in {table_name}"
    print(f"✓ {table_name.capitalize()}: Strict article-level isolation asserted ({len(train_arts_re)} train, {len(val_arts_re)} val, {len(test_arts_re)} test)")

# 3. Verify span resolutions on a random sample of reloaded relations
sample_rels = df_rel_re.sample(n=50, random_state=42)
for _, r in sample_rels.iterrows():
    sec_file = BIOCAUSE_RAW_DIR / f"{r['section_id']}.txt"
    with open(sec_file, "r", encoding="utf-8") as f:
        txt = f.read()
    c_txt = txt[int(r["cause_start"]):int(r["cause_end"])]
    e_txt = txt[int(r["effect_start"]):int(r["effect_end"])]
    assert c_txt == r["cause_text"] or c_txt.replace("\n", " ") == r["cause_text"], "Cause span mismatch!"
    assert e_txt == r["effect_text"] or e_txt.replace("\n", " ") == r["effect_text"], "Effect span mismatch!"
print("✓ BioCause offset integrity: 50 randomly sampled relations resolved exactly against raw text.")

# 4. Reload and verify Corr2Cause features
df_tv_feat_re = pd.read_csv(PROCESSED_DIR / "train_val_features.csv")
df_test_feat_re = pd.read_csv(PROCESSED_DIR / "test_features.csv")
assert len(df_tv_feat_re) == EXPECTED_TRAIN_VAL_SIZE
assert len(df_test_feat_re) == EXPECTED_TEST_SIZE
for col in FEATURE_COLS:
    assert col in df_tv_feat_re.columns and col in df_test_feat_re.columns
    assert df_tv_feat_re[col].isnull().sum() == 0
    assert df_test_feat_re[col].isnull().sum() == 0
print("✓ Corr2Cause feature integrity: All 19 features present with 0 NaNs.")


✓ Relations: Strict article-level isolation asserted (12 train, 4 val, 4 test)
✓ Entities: Strict article-level isolation asserted (12 train, 4 val, 4 test)
✓ Candidates: Strict article-level isolation asserted (12 train, 4 val, 4 test)
✓ BioCause offset integrity: 50 randomly sampled relations resolved exactly against raw text.


✓ Corr2Cause feature integrity: All 19 features present with 0 NaNs.


## 8. Summary and Downstream Contracts for AML_02–AML_04

### Accomplishments in AML_01
1. **Primary Dataset (BioCause):**
   - Faithfully parsed all 198 section files from 20 PMC articles under CC BY-NC-SA 3.0 license.
   - Verified 100% character span resolution against source text.
   - Enforced deterministic, zero-leakage article-level group splitting (12 train, 4 val, 4 test).
   - Exported standardized tables: `biocause_relations.csv` (852 relations), `biocause_entities.csv` (12,939 mentions), and `biocause_candidates.csv` (13,408 candidate pairs).
2. **Supplementary Benchmark (Corr2Cause):**
   - Preserved as an entailment classification benchmark without conflating it with relation extraction.
   - Engineered 19 linguistic and structural features; exported `train_val_features.csv` and `test_features.csv`.

### Contracts for Subsequent Notebooks
- **`AML_02_ner.ipynb`**: Evaluates entity extraction against `biocause_entities.csv` and implements scientific mention extraction.
- **`AML_03_dependency_patterns.ipynb`**: Evaluates syntactic dependency and rule-based relation extraction against `biocause_relations.csv`.
- **`AML_04_classical_classifier.ipynb`**: Trains classical baselines on `biocause_candidates.csv`, builds provenance-aware DAGs, and runs illustrative Bayesian network queries via `pgmpy`.
